# G1 multi-tarefa — sessão isolada

Ambiente **próprio**, separado de qualquer notebook da Lift.

| o quê | onde |
|---|---|
| repo | `/kaggle/input/.../g1-multitask/` (dataset privado, read-only) |
| cópia gravável | `/kaggle/working/g1_mt/` |
| logs e checkpoints | `/kaggle/working/g1_mt/logs/` |
| task registrada | `Mjlab-Multitask-Unitree-G1` |

Nada aqui toca `g1_training/` — o pacote `g1_multitask/` só importa de lá.

**Ordem das células, e o motivo de cada uma:**

1. ambiente — vê o que a Kaggle deu, ANTES de instalar nada
2. instalar — deps pinadas no que roda local
3. verificar — torch/CUDA sobreviveram ao pip? é aqui que se perde a GPU sem perceber
4. montar o repo numa cópia gravável, e só então importar a task
5. simulação do currículo — segundos, sem física
6. pré-voo em GPU
7. treino
8. resume — o portão do plano
9. relatório entre blocos
10. TensorBoard

⚠️ **Rode na ordem, de cima para baixo.** As células 2 a 4 constroem estado no kernel.

## 1. Ambiente, antes de instalar nada

In [ ]:
import subprocess, sys, os, pathlib, shutil, importlib

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "sem GPU: Settings -> Accelerator -> GPU T4 x1"
print("python", sys.version.split()[0])


def acha_repo():
    """Raiz do repo dentro de /kaggle/input, onde quer que a Kaggle a tenha posto.

    `rglob` e nao caminho fixo porque a Kaggle monta ora em `/kaggle/input/<slug>/`,
    ora em `/kaggle/input/datasets/<usuario>/<slug>/`."""
    for base in sorted(pathlib.Path("/kaggle/input").glob("*")):
        achou = list(base.rglob("g1_multitask/kaggle/requirements.txt"))
        if achou:
            return achou[0].parent.parent.parent
    raise SystemExit(
        "dataset nao encontrado em /kaggle/input.\n"
        "  painel direito -> Input -> Add Input -> dataset g1-multitask")


# torch NAO e importado aqui DE PROPOSITO. Se o pip trocar o build na celula de
# instalacao, o kernel ficaria com o antigo carregado — e `importlib.reload(torch)`
# NAO conserta: torch registra operadores C++ no import, e recarregar levanta
# "Only a single TORCH_LIBRARY can be used to register the namespace triton".
print("repo no dataset:", acha_repo())

## 2. Instalar

O mjlab declara a árvore inteira (inclusive `rsl-rl-lib==5.4.0` exato), então instalar
ele puxa o resto. `torch` fica de fora de propósito: a Kaggle já traz um build casado
com o CUDA da imagem, e deixar o pip trocá-lo é o jeito mais rápido de perder a GPU.

In [ ]:
REQ = acha_repo() / "g1_multitask" / "kaggle" / "requirements.txt"
print("requirements:", REQ)
!pip install -q --no-warn-conflicts -r {REQ}

## 3. Verificar — é aqui que se descobre a GPU perdida

Se o pip trocou o torch por um build sem CUDA, tudo abaixo roda em CPU sem reclamar e
a sessão vira 12 h de nada.

A checagem roda num **subprocesso**, não com `importlib.reload(torch)`: torch registra
operadores C++ no import, e recarregar levanta `Only a single TORCH_LIBRARY can be used
to register the namespace triton`.

⚠️ **Esta célula NÃO toca `sys.path` nem importa `g1_multitask`.** Quem faz isso é a
célula 4, e só depois de o repo existir em disco. Inserir em `sys.path` um diretório
que ainda não existe envenena o `sys.path_importer_cache` e o import falha depois com
`No module named 'g1_multitask'` mesmo com o pacote no lugar.

In [ ]:
# Interpretador NOVO = leitura verdadeira do que ficou instalado, sem mexer no
# processo atual. E o jeito certo de checar: reload de torch nao existe.
chk = subprocess.run([sys.executable, "-c",
    "import torch;print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-500:])
assert " True " in f" {chk.stdout} ", (
    "o pip trocou o torch e a CUDA foi embora — reinstale com --no-deps os pacotes "
    "que puxaram torch, ou reinicie o kernel e rode a partir da celula 1")

# so agora importa no kernel, e e a PRIMEIRA vez neste processo
import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
import mjlab, mujoco, warp
print("mujoco", mujoco.__version__, "| warp", warp.config.version)

## 4. Montar o repo numa cópia gravável

`/kaggle/input/` é read-only e o treino escreve `logs/` relativo ao diretório corrente,
então o repo tem que morar em `/kaggle/working/`.

In [ ]:
FONTE = acha_repo()
RAIZ = pathlib.Path("/kaggle/working/g1_mt")

# ⚠️ NAO apagar RAIZ inteira. `logs/` guarda os checkpoints, e a secao 8 retoma deles
# com `--agent.resume True`. Um `rmtree(RAIZ)` aqui leva o curriculo junto e o resume
# recomeca do nivel 0 EM SILENCIO. Limpa so o CODIGO.
PRESERVA = {"logs", "runs", "outputs"}
IGNORA = shutil.ignore_patterns("__pycache__", "*.pyc", ".venv", "logs", "runs")
RAIZ.mkdir(parents=True, exist_ok=True)
for p in RAIZ.iterdir():
    if p.name in PRESERVA:
        continue
    if p.is_dir():
        shutil.rmtree(p)
    else:
        p.unlink()
for p in FONTE.iterdir():
    if p.name in PRESERVA or p.name in (".venv", "__pycache__"):
        continue
    if p.is_dir():
        shutil.copytree(p, RAIZ / p.name, ignore=IGNORA)
    else:
        shutil.copy2(p, RAIZ / p.name)

assert (RAIZ / "g1_multitask" / "__init__.py").is_file(), (
    f"copia incompleta — topo de RAIZ: {sorted(q.name for q in RAIZ.iterdir())}")
os.chdir(RAIZ)
print("raiz:", RAIZ)
print("topo:", sorted(q.name for q in RAIZ.iterdir()))

# ⚠️ `invalidate_caches` e OBRIGATORIO, nao higiene. O Python guarda em
# `sys.path_importer_cache` um finder POR DIRETORIO de `sys.path`, e o finder de um
# diretorio que nao existia no momento da insercao fica cacheado como vazio. Sem esta
# linha, `import g1_multitask` falha com `No module named` mesmo com o pacote em disco.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

import g1_multitask
from mjlab.tasks.registry import list_tasks
print("\ntask registrada:", g1_multitask.TASK_ID in list_tasks())
print("tasks visiveis:", [t for t in list_tasks() if "Unitree-G1" in t])

## 5. Simulação do currículo — segundos, sem física

Prova que a sequência de destravamentos do código é a do desenho: 54 no total, cadeia
de profundidade 9, cada eixo esgotado. **Se isto falhar, não submeta treino.**

In [ ]:
!cd /kaggle/working/g1_mt && python g1_multitask/sim_curriculo.py 2>&1 | tail -25

## 6. Pré-voo em GPU — **o item 0**

`dr.body_com_offset` **corrompe a heap no backend CPU do warp** (medido 30/07: core
dump, e derruba a task do próprio fabricante do mesmo jeito). Ele fica LIGADO no config
porque o treino roda em GPU, onde o caminho de kernel é outro — mas isso nunca foi
verificado. É agora.

Se esta célula derrubar o processo: `DR(base_com=False)` no config, e segue. Perde-se
±2,5 cm de randomização de CoM, não se perde a run.

In [ ]:
!cd /kaggle/working/g1_mt && python g1_multitask/preflight_gpu.py 4096

## 7. Treino — 1000 iterações

**Uma GPU só**, mesmo tendo duas: com dual T4 o `torchrunx` manda o stdout dos workers
para arquivo e as linhas `[CURRICULO]` desaparecem da saída da célula — justamente o
que você quer ver.

`num_envs` é **por rank**. Com uma GPU, 4096 é 4096.

**O que olhar nesta rodada** (as mudanças de 06/08):

| o quê | onde | o que confirma |
|---|---|---|
| equalização de orçamento | dispersão de `Vantagem/std_<tarefa>` | era 0,79 / 1,11 / 1,15; tem de encolher |
| `lift_ao_peito` | `Contrib/pegar/lift` | > 0 e crescendo; era constante ou `nan` |
| `botar` a 4× com `f = 0,5` | `Contrib/botar/box_at_prateleira` + sucesso do `botar` | recompensa alta **com** sucesso |
| `terminacao = −200` | comprimento médio de episódio | se não subir, os −4,0 reais são fracos |

In [ ]:
!cd /kaggle/working/g1_mt && python g1_multitask/train.py \
    --gpu-ids "[0]" \
    --env.scene.num-envs 4096 \
    --agent.max-iterations 1000 \
    2>&1 | grep -vE "^Module |took .* ms" | tail -60

## 8. Resume — **o portão do plano**

Com a run fatiada em blocos de 2k–3k, `save`/`load` dispara de 10 a 15 vezes. Um bug
aqui perde o currículo em **silêncio**: o treino segue rodando, só volta pro nível 0.

Procure a linha:

```
[CURRICULO] retomado: N/54 eventos, M tarefas abertas, push nível K
```

In [ ]:
!cd /kaggle/working/g1_mt && python g1_multitask/train.py \
    --gpu-ids "[0]" \
    --env.scene.num-envs 4096 \
    --agent.max-iterations 100 \
    --agent.resume True \
    2>&1 | grep -E "CURRICULO|Loading model|resume|Error|Traceback" | head -20

## 9. Relatório entre blocos

In [ ]:
!cd /kaggle/working/g1_mt && python g1_multitask/entre_blocos.py 2>&1 | head -80

## 10. TensorBoard (opcional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/g1_mt/logs